# Practical 24 — Natural Background Radiation & Gaussian Distribution

**Institution:** Tri-Chandra Multiple Campus, Tribhuvan University, Kathmandu, Nepal  
**Department:** Physics | B.Sc. III Year | 2082  
**Location:** Rooftop, Tri-Chandra Multiple Campus, Kathmandu  
**Author:** Nabin Pandey | `nabin.795401@trc.tu.edu.np`

---

## Objective

To study the level of natural background radiation in the outdoor field in all directions (east, west, north, south, up and down). Find the standard error in all directions separately. Compile this database in a single set and make a histogram showing Gaussian-like distribution. Interpret the result.

---

## 📐 Theory

### Sources of Natural Background Radiation

Natural background radiation comes from three main sources:

1. **Cosmic rays** — High-energy particles from outer space. They produce secondary particles (muons, electrons) in the atmosphere. Generally stronger in the upward direction.
2. **Terrestrial radiation** — Radioactive isotopes in rocks and soil: U-238, Th-232, K-40. Contribute more in downward/horizontal directions.
3. **Atmospheric radon (Rn-222)** — Released from soil through the U-238 decay chain. Diffuses into the air and contributes in all directions.

### Statistical Nature of Radioactive Decay

Radioactive decay is a **random quantum mechanical process**. The number of counts $N$ in a fixed time interval follows the **Poisson distribution**:

$$P(k; \mu) = \frac{\mu^k \cdot e^{-\mu}}{k!}$$

For a Poisson distribution: $\text{Mean} = \mu$, $\text{Variance} = \mu$, $\sigma = \sqrt{\mu}$

When $\mu \gtrsim 5$, the Poisson distribution is well approximated by a **Gaussian (Normal) distribution**:

$$P(x) = \frac{1}{\sigma\sqrt{2\pi}} \exp\left(-\frac{(x - \mu)^2}{2\sigma^2}\right)$$

This is why we expect background radiation counts to show a **bell-curve (Gaussian-like) histogram** — a direct experimental confirmation of quantum statistical mechanics.

### Statistical Quantities

$$\text{Standard Deviation: } \sigma = \sqrt{\frac{\sum(x_i - \bar{x})^2}{N-1}}$$

$$\text{Standard Error: } SE = \frac{\sigma}{\sqrt{N}}$$

$$\text{Probable Error: } PE = 0.6745 \cdot \sigma$$

PE is the 50% confidence interval — there is a 50% probability that any single measurement lies within $\bar{x} \pm PE$.

---
## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.stats import norm, poisson
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.rcParams['figure.dpi'] = 130

---
## 2. Load Data

100 readings per direction × 6 directions = 600 measurements (599 after removing one missing entry in west column).

Each reading = GM count over **10 seconds** at rooftop of Tri-Chandra Multiple Campus.

In [ ]:
df = pd.read_csv('data.csv')

print(f'Shape: {df.shape}')
print(f'\nMissing values per direction:')
print(df.isna().sum())

df.head(10)

---
## 3. Statistics Per Direction

For each direction, compute mean, standard deviation, standard error, and probable error.

In [ ]:
directions = ['down', 'up', 'north', 'east', 'south', 'west']

stats = {}
for d in directions:
    clean = df[d].dropna().values.astype(float)   # drop NaN (west row 52)
    n    = len(clean)
    mean = clean.mean()
    std  = clean.std(ddof=1)          # Bessel-corrected sample std dev
    se   = std / np.sqrt(n)           # standard error of the mean
    pe   = 0.6745 * std               # probable error (50% CI)
    stats[d] = {'n': n, 'mean': mean, 'std': std, 'se': se, 'pe': pe, 'data': clean}

# Display as a clean table
summary = pd.DataFrame({
    'Direction': [d.capitalize() for d in directions],
    'n': [stats[d]['n'] for d in directions],
    'Mean': [stats[d]['mean'] for d in directions],
    'Std Dev (σ)': [stats[d]['std'] for d in directions],
    'Std Error (SE)': [stats[d]['se'] for d in directions],
    'Probable Error (PE)': [stats[d]['pe'] for d in directions],
})

summary.style.format({
    'Mean': '{:.4f}', 'Std Dev (σ)': '{:.4f}',
    'Std Error (SE)': '{:.4f}', 'Probable Error (PE)': '{:.4f}'
}).set_caption('Statistical Summary — Per Direction (Counting time: 10 s)')

---
## 4. Combined Dataset Statistics

In [ ]:
# Compile all directions into one flat array
combined = np.concatenate([stats[d]['data'] for d in directions])

mu_c  = combined.mean()
std_c = combined.std(ddof=1)
se_c  = std_c / np.sqrt(len(combined))
pe_c  = 0.6745 * std_c

print("╔══════════════════════════════════════════════════════════╗")
print("║       COMBINED DATASET — ALL DIRECTIONS                  ║")
print("╠══════════════════════════════════════════════════════════╣")
print(f"║  Total measurements  n  =  {len(combined)}                       ║")
print(f"║  Mean count rate     μ  =  {mu_c:.4f} counts / 10 s      ║")
print(f"║  Standard deviation  σ  =  {std_c:.4f}                    ║")
print(f"║  Standard error      SE =  {se_c:.4f}                    ║")
print(f"║  Probable error      PE =  {pe_c:.4f}                    ║")
print("╠══════════════════════════════════════════════════════════╣")
print(f"║  Poisson check: σ²/μ = {std_c**2:.4f}/{mu_c:.4f} = {std_c**2/mu_c:.4f}        ║")
print(f"║  (Should be ≈ 1.0 for pure Poisson process)              ║")
print("╚══════════════════════════════════════════════════════════╝")

---
## 5. Visualisation

**6 directional histograms** with Gaussian fit and Poisson PMF overlay, plus one **combined histogram** for the full dataset.

In [ ]:
BG     = '#0d1117'; PANEL  = '#161b22'; BORDER = '#30363d'; GRID   = '#21262d'
BLUE   = '#58a6ff'; ORANGE = '#f0883e'; GREEN  = '#56d364'; PURPLE = '#bc8cff'
YELLOW = '#e3b341'; MUTED  = '#8b949e'; WHITE  = '#e6edf3'
COLORS = [BLUE, ORANGE, GREEN, PURPLE, YELLOW, '#ff7b72']

fig = plt.figure(figsize=(16, 11), facecolor=BG)
gs  = gridspec.GridSpec(3, 3, figure=fig,
                        hspace=0.52, wspace=0.38,
                        left=0.08, right=0.96, top=0.91, bottom=0.07)

def style_ax(ax, title=''):
    ax.set_facecolor(PANEL)
    for sp in ax.spines.values(): sp.set_edgecolor(BORDER)
    ax.tick_params(colors=MUTED, labelsize=8.5)
    ax.xaxis.label.set_color(MUTED); ax.yaxis.label.set_color(MUTED)
    ax.grid(True, color=GRID, linewidth=0.6, linestyle='--', alpha=0.8)
    if title: ax.set_title(title, color=WHITE, fontsize=10, fontweight='bold', pad=7)

# ── Panels 1–6: per direction ─────────────────────────────────────────────
for i, d in enumerate(directions):
    row, col = divmod(i, 3)
    ax   = fig.add_subplot(gs[row, col])
    data = stats[d]['data']
    mu_d = stats[d]['mean']
    sd_d = stats[d]['std']

    bins = np.arange(data.min() - 0.5, data.max() + 1.5, 1)
    ax.hist(data, bins=bins, density=True, color=COLORS[i],
            alpha=0.55, edgecolor=BORDER, linewidth=0.6, label='Observed')

    x_g = np.linspace(data.min() - 1, data.max() + 1, 300)
    ax.plot(x_g, norm.pdf(x_g, mu_d, sd_d), color=WHITE, lw=1.8, label='Gaussian')

    k_vals = np.arange(0, int(data.max()) + 2)
    ax.plot(k_vals, poisson.pmf(k_vals, mu_d), 'o--',
            color=YELLOW, markersize=3.5, lw=1.2, alpha=0.85, label='Poisson')

    ann = (f'μ = {mu_d:.2f}\nσ = {sd_d:.2f}\nSE = {stats[d]["se"]:.4f}\nPE = {stats[d]["pe"]:.4f}')
    ax.text(0.97, 0.96, ann, transform=ax.transAxes, fontsize=7.8,
            va='top', ha='right', color=WHITE, fontfamily='monospace',
            bbox=dict(boxstyle='round,pad=0.4', facecolor='#21262d',
                      edgecolor=COLORS[i], alpha=0.9))
    ax.set_xlabel('Counts per 10 s', fontsize=8.5)
    ax.set_ylabel('Probability Density', fontsize=8.5)
    style_ax(ax, f'{d.capitalize()} Direction')
    leg = ax.legend(fontsize=7.5, facecolor=PANEL, edgecolor=BORDER)
    for t in leg.get_texts(): t.set_color(WHITE)

# ── Panel 7: combined ─────────────────────────────────────────────────────
ax7    = fig.add_subplot(gs[2, :])
bins_c = np.arange(combined.min() - 0.5, combined.max() + 1.5, 1)
ax7.hist(combined, bins=bins_c, density=True, color=BLUE, alpha=0.5,
         edgecolor=BORDER, linewidth=0.7, label=f'All directions combined (n={len(combined)})')

x_gc = np.linspace(combined.min() - 1, combined.max() + 1, 400)
ax7.plot(x_gc, norm.pdf(x_gc, mu_c, std_c), color=ORANGE, lw=2.3,
         label=f'Gaussian: μ={mu_c:.4f}, σ={std_c:.4f}')

k_c = np.arange(0, int(combined.max()) + 2)
ax7.plot(k_c, poisson.pmf(k_c, mu_c), 'o--', color=GREEN,
         markersize=4, lw=1.4, alpha=0.85, label=f'Poisson: λ={mu_c:.4f}')

ax7.axvline(mu_c, color=WHITE, lw=1.2, linestyle=':', alpha=0.7,
            label=f'Mean = {mu_c:.4f}')
ax7.axvspan(mu_c - std_c, mu_c + std_c, color=ORANGE, alpha=0.07,
            label='±1σ  (68.3% of data)')

ann7 = (f'μ = {mu_c:.4f}   σ = {std_c:.4f}\n'
        f'SE = {se_c:.4f}   PE = {pe_c:.4f}\n'
        f'n = {len(combined)}  (6 directions × ~100 readings each)')
ax7.text(0.98, 0.95, ann7, transform=ax7.transAxes, fontsize=9,
         va='top', ha='right', color=WHITE, fontfamily='monospace',
         bbox=dict(boxstyle='round,pad=0.5', facecolor='#21262d',
                   edgecolor=ORANGE, alpha=0.92))
ax7.set_xlabel('Counts per 10 s  (natural background radiation)', fontsize=10)
ax7.set_ylabel('Probability Density', fontsize=10)
style_ax(ax7, 'Combined Dataset — Gaussian-like Distribution of Natural Background Radiation')
leg7 = ax7.legend(fontsize=9, facecolor=PANEL, edgecolor=BORDER)
for t in leg7.get_texts(): t.set_color(WHITE)

fig.suptitle(
    'Practical 24 — Natural Background Radiation · Angular Distribution · '
    'GM Counter · Rooftop, Tri-Chandra Campus, Kathmandu',
    fontsize=12, fontweight='bold', color=WHITE, y=0.975)
fig.text(0.5, 0.003,
         'Tri-Chandra Multiple Campus · Department of Physics · B.Sc. III Year · 2082',
         ha='center', fontsize=8.5, color=MUTED)

plt.savefig('practical24_figure.png', dpi=180, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()

---
## 6. Results Table

In [ ]:
result_rows = []
for d in directions:
    s = stats[d]
    result_rows.append({
        'Direction': d.capitalize(),
        'n': s['n'],
        'Mean (counts/10s)': round(s['mean'], 4),
        'Std Dev σ': round(s['std'], 4),
        'Std Error SE': round(s['se'], 4),
        'Probable Error PE': round(s['pe'], 4),
    })

# Add combined row
result_rows.append({
    'Direction': 'Combined',
    'n': len(combined),
    'Mean (counts/10s)': round(mu_c, 4),
    'Std Dev σ': round(std_c, 4),
    'Std Error SE': round(se_c, 4),
    'Probable Error PE': round(pe_c, 4),
})

result_df = pd.DataFrame(result_rows)
result_df.style\
    .set_caption('Results — Natural Background Radiation (Counting time: 10 s per reading)')\
    .apply(lambda x: ['font-weight: bold' if x['Direction'] == 'Combined'
                      else '' for _ in x], axis=1)

---
## 7. Discussion & Interpretation

### Is the distribution Gaussian?

Yes. The combined histogram of 599 measurements clearly shows a **bell-shaped (Gaussian-like) distribution** centred around μ ≈ 4.55 counts per 10 seconds. This is expected from the **Central Limit Theorem** — when many independent random events are summed, the result converges to a Gaussian, even if the underlying process is Poisson.

### Is the distribution isotropic?

The mean count rates across all six directions range from 4.37 to 4.85 counts/10s — a spread of only ~0.5 counts. Given the standard errors (0.18–0.21), these means are **statistically consistent** with each other. This means background radiation at the rooftop of Tri-Chandra campus is approximately **isotropic** — no significant directional preference.

### Poisson check

For a Poisson process, variance = mean (σ²/μ = 1). Here σ²/μ = 0.863 ≈ 1. This is good agreement — the slight underdispersion may reflect very stable measurement conditions on the rooftop.

### Physical insight

This experiment is a direct experimental verification that **quantum mechanical randomness produces statistically predictable distributions**. Every GM click is a single quantum event — individually unpredictable, collectively tracing a perfect Gaussian. That is one of the most profound results in physics.

---
## 8. Conclusion

| Quantity | Value |
|---|---|
| Mean background count rate | **4.5476 counts / 10 s** |
| Standard deviation σ | **1.9808** |
| Standard error SE | **0.0809** |
| Probable error PE | **1.3361** |
| Poisson check σ²/μ | **0.863** (≈ 1 ✅) |

The combined histogram confirms a **Gaussian-like distribution**, validating the statistical nature of radioactive decay. Background radiation at Tri-Chandra campus rooftop is isotropic, with no statistically significant directional dependence.

---
## 9. Sources of Error

| Source | Type | Effect |
|---|---|---|
| Statistical fluctuations in counts | Random | Inherent to radioactive decay — reduced by large N |
| GM tube dead time | Systematic | Slight undercounting at higher rates |
| Directional sensitivity of GM tube | Systematic | Tube geometry may favour certain angles |
| Nearby building materials | Systematic | Concrete/brick adds terrestrial radiation |
| Cosmic ray flux variation | Systematic | Slowly varies with atmospheric pressure |
| Missing data point (west, row 52) | Systematic | Removed via dropna() — negligible effect |

---
*Nabin Pandey | B.Sc. Physics, Tri-Chandra Multiple Campus, TU | 2082*  
*`nabin.795401@trc.tu.edu.np` | [LinkedIn](https://np.linkedin.com/in/nabinpandey1661)*